# Contact coordinates: design (.oas) and SEM (.jpg)

Prints the contact centres from one `.oas` file and its matching SEM image.

| Output | Function | Units | (0, 0) is… | y points |
|---|---|---|---|---|
| Design `x_nm, y_nm` | `read_contacts` (polygon centroid) | nm | the `.oas` file's own origin (values as stored, no shift) | up |
| SEM `x_px, y_px` | `detect_contacts` (Otsu, region centroid) | px | centre of the top-left pixel | **down** |
| SEM `x_tile_nm, y_tile_nm` | `pixel_to_tile_nm` (needs `PIXEL_SIZE_NM`) | nm | centre of the image (after removing the data bar) | up |

The design and SEM coordinates are not aligned to each other yet. The SEM tile's position on
the mask will come from its stage coordinates (metadata, not read yet).

Set the paths below and run all cells.

In [ ]:
OAS_PATH = r"path\to\tile.oas"
JPG_PATH = r"path\to\tile.jpg"
LAYER = (1, 0)         # (layer, datatype) of the contacts
CELL = None            # top cell name; None if the file has a single top cell
DATABAR_ROWS = 0       # rows of SEM info bar at the bottom of the image to ignore
DARK_CONTACTS = True   # True if contacts look dark in the SEM image
MIN_AREA_PX = 20       # ignore regions smaller than this many pixels
PIXEL_SIZE_NM = None   # nm per pixel; set it to also get tile-local nm

In [ ]:
import pandas as pd

from affine_ransac.features.contact import detect_contacts
from affine_ransac.geometry.frames import pixel_to_tile_nm
from affine_ransac.io.design import list_layers, load_layout, read_contacts
from affine_ransac.io.sem_image import crop_databar, load_sem_image

## Design contacts (.oas)

In [ ]:
layout = load_layout(OAS_PATH)
print(f"Database unit: {layout.dbu * 1000:g} nm")
print(f"Layers: {list_layers(layout)}")

centers_nm, sizes_nm = read_contacts(layout, *LAYER, cell_name=CELL)
design = pd.DataFrame({
    "x_nm": centers_nm[:, 0], "y_nm": centers_nm[:, 1],
    "w_nm": sizes_nm[:, 0], "h_nm": sizes_nm[:, 1],
})
print(f"{len(design)} contacts")
design

## SEM contacts (.jpg)

In [ ]:
image = crop_databar(load_sem_image(JPG_PATH), DATABAR_ROWS)
print(f"Image used: {image.shape[1]} x {image.shape[0]} px (after removing {DATABAR_ROWS} data-bar rows)")

found = detect_contacts(image, dark_contacts=DARK_CONTACTS, min_area_px=MIN_AREA_PX)
sem = pd.DataFrame({"x_px": found.centers[:, 0], "y_px": found.centers[:, 1], "area_px": found.areas})
if PIXEL_SIZE_NM:
    xy = pixel_to_tile_nm(found.centers, image.shape, PIXEL_SIZE_NM)
    sem["x_tile_nm"], sem["y_tile_nm"] = xy[:, 0], xy[:, 1]

print(f"Otsu threshold: {found.threshold:.0f}")
print(f"{len(sem)} contacts")
sem

## Visual check (optional)

Opens two interactive windows side by side: the design (polygons with centroid `+`) and the
SEM image (detected contours in green, centres as red `+`). The cell waits until you close
both windows.

In [ ]:
import pyqtgraph as pg
from affine_ransac import view_design, view_sem

app = pg.mkQApp()
design_win = view_design.build_window(OAS_PATH, LAYER, CELL)
sem_win = view_sem.build_window(JPG_PATH, found)
design_win.show()
sem_win.show()
app.exec()